In [1]:
from pathlib import Path
from update_para_files import modify_phot_para_many
import subprocess

import pandas as pd
import numpy as np
import os

#modify_phot_para_many?

# Galaxy fitting

## Setting the paths of the inputs

In [2]:
# This is the directory where you put all the files you need
root_dir = Path(".").resolve()

# the para file for zphota program
zphot_galaxy_path = root_dir / "zphot_galaxy.para"

# photometry data
catalog_path = root_dir / "magnitudes.sed"

# the output file that defines the output columns
output_para_path = root_dir / "zphot_output.para"

# the output file including (best fit, error, chi^2, PDF, etc.)
out_dir = root_dir / "galaxy_model_fitting"
out_dir.mkdir(parents=True, exist_ok=True)

output_path = out_dir / "galaxy_GAL.out"

In [3]:
updates = [
    ("CAT_IN", catalog_path), 
    ("CAT_OUT", output_path), 
    ("PARA_OUT", output_para_path),
]

modify_phot_para_many(zphot_galaxy_path, 
                      updates = updates, 
                      replace_all = False,
                      strict = True, 
                      backup = True,)

# you can use git diff --no-index zphot_galaxy.para.bak zphot_galaxy.para to check the difference of the two files

{'CAT_IN': True, 'CAT_OUT': True, 'PARA_OUT': True}

## Start fitting

In [4]:
##############################################################################
#                CREATION OF LIBRARIES FROM SEDs List                        #
# $LEPHAREDIR/source/sedtolib -t (S/Q/G) -c $LEPHAREDIR/config/zphot.para    #
# help : $LEPHAREDIR/source/sedtolib -h (or -help)                           #
##############################################################################

In [5]:
# build the star sed bin library
subprocess.run(f'$LEPHAREDIR/source/sedtolib -t S -c {zphot_galaxy_path}', check=True, shell=True)

    
  first pass : reading each SED ...
############################################
#  It s translating SEDs to binary library #
#     with the following options :          
# Config file     : /media/psf/Home/Desktop/ResearchCodes/Projects/Photoz/LePHARE_Fitting_Fortran/zphot_galaxy.para
# Library type    : S
# Number of SEDs  :    254
# STAR_SED    :/home/shengyong/LePhare/lephare_dev/sed/STAR/STAR_MOD_ALL.list
# STAR_LIB    :/home/shengyong/LePhare/lepharework/lib_bin/LIB_STAR.bin
# STAR_LIB doc:/home/shengyong/LePhare/lepharework/lib_bin/LIB_STAR.doc
# STAR_WMAX   :    7776
# STAR_FSCALE :0.343200E-08
#######################################
  writing in binary library ...
  DONE            extGen.7.sed           7776 points per record 


CompletedProcess(args='$LEPHAREDIR/source/sedtolib -t S -c /media/psf/Home/Desktop/ResearchCodes/Projects/Photoz/LePHARE_Fitting_Fortran/zphot_galaxy.para', returncode=0)

In [6]:
# build the galaxy sed bin library
subprocess.run(f'$LEPHAREDIR/source/sedtolib -t G -c {zphot_galaxy_path}', check=True, shell=True)

    
  first pass : reading each SED ...
############################################
#  It s translating SEDs to binary library #
#     with the following options :          
# Config file     : /media/psf/Home/Desktop/ResearchCodes/Projects/Photoz/LePHARE_Fitting_Fortran/zphot_galaxy.para
# Library type    : G
# Number of SEDs  :     30
# GAL_SED    :/home/shengyong/LePhare/lephare_dev/sed/GAL/MARA23032010_MOD.list
# GAL_LIB    :/home/shengyong/LePhare/lepharework/lib_bin/LIB_COSMOS.bin
# GAL_LIB doc:/home/shengyong/LePhare/lepharework/lib_bin/LIB_COSMOS.doc
# GAL_LIB phys:/home/shengyong/LePhare/lepharework/lib_bin/LIB_COSMOS.phys
# SEL_AGE    :NONE
# GAL_WMAX   :    4474
# GAL_FSCALE :0.100000E+01
# AGE_RANGE    0.000000E+00 0.130000E+11
#######################################
  writing in binary library ...
  DONE            SO1_template_norm.sed   747 points per record 


CompletedProcess(args='$LEPHAREDIR/source/sedtolib -t G -c /media/psf/Home/Desktop/ResearchCodes/Projects/Photoz/LePHARE_Fitting_Fortran/zphot_galaxy.para', returncode=0)

In [7]:
#############################################################################
#                           FILTERS                                         #
#  $LEPHAREDIR/source/filter  -c $LEPHAREDIR/config/zphot.para              #
#  help: $LEPHAREDIR/source/filter  -h (or -help)                           #
#############################################################################

In [8]:
# build the ASCII filterset file
subprocess.run(f'$LEPHAREDIR/source/filter -c {zphot_galaxy_path}', check=True, shell=True)

#####################################
# It s building the filter file     #
#   with the following options :    #
# Config file : /media/psf/Home/Desktop/ResearchCodes/Projects/Photoz/LePHARE_Fitting_Fortran/zphot_galaxy.para
# TRANS_TYPE  :  0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
# FILTER_CALIB:   0  0  0  0  0  0  0  0  0  0
# FILTER_FILE : /home/shengyong/LePhare/lepharework/filt/sdss_ugriz.filt
# FILTER_FILE.doc: /home/shengyong/LePhare/lepharework/filt/sdss_ugriz.doc
# FILTER_LIST : sdss/gp.pb sdss/rp.pb sdss/ip.pb sdss/zp.pb uvot/uvw2.pb uvot/uvm2.pb uvot/uvw1.pb uvot/u.pb uvot/b.pb uvot/v.pb 
#####################################
#     89              gp     0    1
#     75              rp     0    2
#     89              ip     0    3
#    139              zp     0    4
#    264            uvw2     0    5
#    205            uvm2     0    6
#    312            uvw1     0    7
#    196               u     0    8
#    272               b     0    9
#    227               v     0   10
# NA

CompletedProcess(args='$LEPHAREDIR/source/filter -c /media/psf/Home/Desktop/ResearchCodes/Projects/Photoz/LePHARE_Fitting_Fortran/zphot_galaxy.para', returncode=0)

In [9]:
############################################################################
#                 THEORETICAL  MAGNITUDES                                  #
# $LEPHAREDIR/source/mag_star -c  $LEPHAREDIR/config/zphot.para (star only)#
# help: $LEPHAREDIR/source/mag_star -h (or -help)                          #
# $LEPHAREDIR/source/mag_gal  -t (Q or G) -c $LEPHAREDIR/config/zphot.para #
#                                                         (for gal. & QSO) #
# help: $LEPHAREDIR/source/mag_gal  -h (or -help)                          #
############################################################################

In [10]:
# build the binary Star theoretical magnitude library

subprocess.run(f'$LEPHAREDIR/source/mag_star -c {zphot_galaxy_path}', check=True, shell=True)

 10
# NAME    IDENT      Lbda_mean    Lbeff(Vega)       FWHM     AB-cor  TG-cor  VEGA  M_sun(AB) CALIB    Lb_eff   Fac_corr
gp           1         0.4719         0.4641         0.1263  -0.085  -0.297 -20.660   5.159   0         0.4719  1.000
rp           2         0.6185         0.6122         0.1150   0.155   0.253 -21.500   4.663   0         0.6185  1.000
ip           3         0.7500         0.7440         0.1239   0.370   0.573 -22.135   4.549   0         0.7500  1.000
zp           4         0.8961         0.8900         0.0994   0.537   0.768 -22.686   4.525   0         0.8961  1.000
uvw2         5         0.2030         0.1996         0.0560   1.893 -99.990 -20.779  11.547   0         0.2030  1.000
uvm2         6         0.2253         0.2229         0.0510   1.788 -99.990 -20.927  10.631   0         0.2253  1.000
uvw1         7         0.2612         0.2587         0.0685   1.641 -99.990 -21.080   8.678   0         0.2612  1.000
u            8         0.3470         0.3497      

CompletedProcess(args='$LEPHAREDIR/source/mag_star -c /media/psf/Home/Desktop/ResearchCodes/Projects/Photoz/LePHARE_Fitting_Fortran/zphot_galaxy.para', returncode=0)

In [11]:
# build the binary Galaxy theoretical magnitude library

subprocess.run(f'$LEPHAREDIR/source/mag_gal -t G -c {zphot_galaxy_path}', check=True, shell=True)

  reading library doc ...
 number of record : 30 60 35824
 reading Physical parameters from
 /home/shengyong/LePhare/lepharework/lib_bin/LIB_COSMOS.phys with 60
# NAME    IDENT      Lbda_mean    Lbeff(Vega)       FWHM     AB-cor  TG-cor  VEGA  M_sun(AB) CALIB    Lb_eff   Fac_corr
gp           1         0.4719         0.4641         0.1263  -0.085  -0.297 -20.660   5.159   0         0.4719  1.000
rp           2         0.6185         0.6122         0.1150   0.155   0.253 -21.500   4.663   0         0.6185  1.000
ip           3         0.7500         0.7440         0.1239   0.370   0.573 -22.135   4.549   0         0.7500  1.000
zp           4         0.8961         0.8900         0.0994   0.537   0.768 -22.686   4.525   0         0.8961  1.000
uvw2         5         0.2030         0.1996         0.0560   1.893 -99.990 -20.779  11.547   0         0.2030  1.000
uvm2         6         0.2253         0.2229         0.0510   1.788 -99.990 -20.927  10.631   0         0.2253  1.000
uvw1       

CompletedProcess(args='$LEPHAREDIR/source/mag_gal -t G -c /media/psf/Home/Desktop/ResearchCodes/Projects/Photoz/LePHARE_Fitting_Fortran/zphot_galaxy.para', returncode=0)

In [12]:
# fit photometric redshifts

subprocess.run(f'$LEPHAREDIR/source/zphota -c {zphot_galaxy_path}', check=True, shell=True)


#######################################
# PHOTOMETRIC REDSHIFT with OPTIONS   #
# CAT_IN       : /media/psf/Home/Desktop/ResearchCodes/Projects/Photoz/LePHARE_Fitting_Fortran/magnitudes.sed
# CAT_OUT      : /media/psf/Home/Desktop/ResearchCodes/Projects/Photoz/LePHARE_Fitting_Fortran/galaxy_model_fitting/galaxy_GAL.out
# CAT_LINES     :        -99        -99
# PARA_OUT     : /media/psf/Home/Desktop/ResearchCodes/Projects/Photoz/LePHARE_Fitting_Fortran/zphot_output.para
# INP_TYPE     : M
# CAT_FMT[0:MEME 1:MMEE]:      0
# CAT_MAG      : AB
# ZPHOTLIB     : STAR_GR+UV LIB_COSMOS_OUT 
# ADD_EMLINES  : NO
# ERR_SCALE    :  0.100  0.100  0.100  0.100  0.100  0.100  0.100  0.100  0.100  0.100
# ERR_FACTOR   :  1.000
# BD_SCALE     :          0
# GLB_CONTEXT  :          0
# CHI2_RM_BD   :  10000.00 10000.00
# Z_RANGE      :    0.000   99.990
# EBV_RANGE  :    0.000    9.000
# DZ_WIN       :  0.500
# MIN_THRES    :  0.100
# MASS_SCALE   :    0.000   0.000
# MAG_ABS      :   -8.000 -30.000
# M

CompletedProcess(args='$LEPHAREDIR/source/zphota -c /media/psf/Home/Desktop/ResearchCodes/Projects/Photoz/LePHARE_Fitting_Fortran/zphot_galaxy.para', returncode=0)

In [13]:
# final cleaning spec files

src_dir = root_dir
dst_dir = out_dir

dst_dir.mkdir(parents=True, exist_ok=True)

for p in src_dir.rglob("I*.spec"):   # 如果只想当前层，用 src_dir.glob("*.spec")
    target = dst_dir / p.name       # 保留文件名
    p.replace(target)               # move

# Powerlaw fitting

## Setting the paths of the inputs

In [14]:
# This is the directory where you put all the files you need
root_dir = Path(".").resolve()

# the para file for zphota program
zphot_powerlaw_path = root_dir / "zphot_powerlaw.para"

# the output file including (best fit, error, chi^2, PDF, etc.)
out_dir = root_dir / "powerlaw_model_fitting"
out_dir.mkdir(parents=True, exist_ok=True)

In [15]:
# photometry data
catalog_path = root_dir / "magnitudes.sed"

# the output file that defines the output columns
output_para_path = root_dir / "zphot_output.para"

output_path = out_dir / "powerlaw_GAL.out"

In [16]:
# update the file paths in the para file

updates = [
    ("CAT_IN", catalog_path), 
    ("CAT_OUT", output_path), 
    ("PARA_OUT", output_para_path),
]

modify_phot_para_many(zphot_powerlaw_path, 
                      updates = updates, 
                      replace_all = False,
                      strict = True, 
                      backup = True,)

# you can use git diff --no-index zphot_galaxy.para.bak zphot_galaxy.para to check the difference of the two files

{'CAT_IN': True, 'CAT_OUT': True, 'PARA_OUT': True}

## Start fitting

In [17]:
#############################################################################
#                           FILTERS                                         #
#  $LEPHAREDIR/source/filter  -c $LEPHAREDIR/config/zphot.para              #
#  help: $LEPHAREDIR/source/filter  -h (or -help)                           #
#############################################################################

In [18]:
# build the star sed bin library
subprocess.run(f'$LEPHAREDIR/source/sedtolib -t S -c {zphot_powerlaw_path}', check=True, shell=True)

    
  first pass : reading each SED ...
############################################
#  It s translating SEDs to binary library #
#     with the following options :          
# Config file     : /media/psf/Home/Desktop/ResearchCodes/Projects/Photoz/LePHARE_Fitting_Fortran/zphot_powerlaw.para
# Library type    : S
# Number of SEDs  :    254
# STAR_SED    :/home/shengyong/LePhare/lephare_dev/sed/STAR/STAR_MOD_ALL.list
# STAR_LIB    :/home/shengyong/LePhare/lepharework/lib_bin/LIB_STAR.bin
# STAR_LIB doc:/home/shengyong/LePhare/lepharework/lib_bin/LIB_STAR.doc
# STAR_WMAX   :    7776
# STAR_FSCALE :0.343200E-08
#######################################
  writing in binary library ...
  DONE            extGen.7.sed           7776 points per record 


CompletedProcess(args='$LEPHAREDIR/source/sedtolib -t S -c /media/psf/Home/Desktop/ResearchCodes/Projects/Photoz/LePHARE_Fitting_Fortran/zphot_powerlaw.para', returncode=0)

In [19]:
# build the powerlaw sed bin library
subprocess.run(f'$LEPHAREDIR/source/sedtolib -t Q -c {zphot_powerlaw_path}', check=True, shell=True)

    
  first pass : reading each SED ...
############################################
#  It s translating SEDs to binary library #
#     with the following options :          
# Config file     : /media/psf/Home/Desktop/ResearchCodes/Projects/Photoz/LePHARE_Fitting_Fortran/zphot_powerlaw.para
# Library type    : Q
# Number of SEDs  :     60
# QSO_SED    :/home/shengyong/LePhare/lephare_dev/sed/QSO/PL/PL_some.list
# QSO_LIB    :/home/shengyong/LePhare/lepharework/lib_bin/LIB_PL_some_GR+UV.bin
# QSO_LIB doc:/home/shengyong/LePhare/lepharework/lib_bin/LIB_PL_some_GR+UV.doc
# QSO_WMAX   :    1266
# QSO_FSCALE :0.100000E+01
#######################################
  writing in binary library ...
  DONE            d                       300 points per record 


CompletedProcess(args='$LEPHAREDIR/source/sedtolib -t Q -c /media/psf/Home/Desktop/ResearchCodes/Projects/Photoz/LePHARE_Fitting_Fortran/zphot_powerlaw.para', returncode=0)

In [20]:
#############################################################################
#                           FILTERS                                         #
#  $LEPHAREDIR/source/filter  -c $LEPHAREDIR/config/zphot.para              #
#  help: $LEPHAREDIR/source/filter  -h (or -help)                           #
#############################################################################

In [21]:
# build the ASCII filterset file
subprocess.run(f'$LEPHAREDIR/source/filter -c {zphot_powerlaw_path}', check=True, shell=True)

#####################################
# It s building the filter file     #
#   with the following options :    #
# Config file : /media/psf/Home/Desktop/ResearchCodes/Projects/Photoz/LePHARE_Fitting_Fortran/zphot_powerlaw.para
# TRANS_TYPE  :  0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
# FILTER_CALIB:   0  0  0  0  0  0  0  0  0  0
# FILTER_FILE : /home/shengyong/LePhare/lepharework/filt/sdss_ugriz.filt
# FILTER_FILE.doc: /home/shengyong/LePhare/lepharework/filt/sdss_ugriz.doc
# FILTER_LIST : sdss/gp.pb sdss/rp.pb sdss/ip.pb sdss/zp.pb uvot/uvw2.pb uvot/uvm2.pb uvot/uvw1.pb uvot/u.pb uvot/b.pb uvot/v.pb 
#####################################
#     89              gp     0    1
#     75              rp     0    2
#     89              ip     0    3
#    139              zp     0    4
#    264            uvw2     0    5
#    205            uvm2     0    6
#    312            uvw1     0    7
#    196               u     0    8
#    272               b     0    9
#    227               v     0   10
# 

CompletedProcess(args='$LEPHAREDIR/source/filter -c /media/psf/Home/Desktop/ResearchCodes/Projects/Photoz/LePHARE_Fitting_Fortran/zphot_powerlaw.para', returncode=0)

In [22]:
############################################################################
#                 THEORETICAL  MAGNITUDES                                  #
# $LEPHAREDIR/source/mag_star -c  $LEPHAREDIR/config/zphot.para (star only)#
# help: $LEPHAREDIR/source/mag_star -h (or -help)                          #
# $LEPHAREDIR/source/mag_gal  -t (Q or G) -c $LEPHAREDIR/config/zphot.para #
#                                                         (for gal. & QSO) #
# help: $LEPHAREDIR/source/mag_gal  -h (or -help)                          #
############################################################################

In [23]:
# build the binary Star theoretical magnitude library

subprocess.run(f'$LEPHAREDIR/source/mag_star -c {zphot_powerlaw_path}', check=True, shell=True)

 10
# NAME    IDENT      Lbda_mean    Lbeff(Vega)       FWHM     AB-cor  TG-cor  VEGA  M_sun(AB) CALIB    Lb_eff   Fac_corr
gp           1         0.4719         0.4641         0.1263  -0.085  -0.297 -20.660   5.159   0         0.4719  1.000
rp           2         0.6185         0.6122         0.1150   0.155   0.253 -21.500   4.663   0         0.6185  1.000
ip           3         0.7500         0.7440         0.1239   0.370   0.573 -22.135   4.549   0         0.7500  1.000
zp           4         0.8961         0.8900         0.0994   0.537   0.768 -22.686   4.525   0         0.8961  1.000
uvw2         5         0.2030         0.1996         0.0560   1.893 -99.990 -20.779  11.547   0         0.2030  1.000
uvm2         6         0.2253         0.2229         0.0510   1.788 -99.990 -20.927  10.631   0         0.2253  1.000
uvw1         7         0.2612         0.2587         0.0685   1.641 -99.990 -21.080   8.678   0         0.2612  1.000
u            8         0.3470         0.3497      

CompletedProcess(args='$LEPHAREDIR/source/mag_star -c /media/psf/Home/Desktop/ResearchCodes/Projects/Photoz/LePHARE_Fitting_Fortran/zphot_powerlaw.para', returncode=0)

In [24]:
# build the binary powerlaw theoretical magnitude library

subprocess.run(f'$LEPHAREDIR/source/mag_gal -t G -c {zphot_powerlaw_path}', check=True, shell=True)

  reading library doc ...
 number of record : 60 120 10160
 reading Physical parameters from
 /home/shengyong/LePhare/lepharework/lib_bin/LIB_PL_some_GR+UV.phys with 120
  Size problem for the  lib_bin library LIB_PL_some_GR+UV 60 120
# NAME    IDENT      Lbda_mean    Lbeff(Vega)       FWHM     AB-cor  TG-cor  VEGA  M_sun(AB) CALIB    Lb_eff   Fac_corr
gp           1         0.4719         0.4641         0.1263  -0.085  -0.297 -20.660   5.159   0         0.4719  1.000
rp           2         0.6185         0.6122         0.1150   0.155   0.253 -21.500   4.663   0         0.6185  1.000
ip           3         0.7500         0.7440         0.1239   0.370   0.573 -22.135   4.549   0         0.7500  1.000
zp           4         0.8961         0.8900         0.0994   0.537   0.768 -22.686   4.525   0         0.8961  1.000
uvw2         5         0.2030         0.1996         0.0560   1.893 -99.990 -20.779  11.547   0         0.2030  1.000
uvm2         6         0.2253         0.2229         0.

CompletedProcess(args='$LEPHAREDIR/source/mag_gal -t G -c /media/psf/Home/Desktop/ResearchCodes/Projects/Photoz/LePHARE_Fitting_Fortran/zphot_powerlaw.para', returncode=0)

In [25]:
# fit photometric redshifts

subprocess.run(f'$LEPHAREDIR/source/zphota -c {zphot_powerlaw_path}', check=True, shell=True)


#######################################
# PHOTOMETRIC REDSHIFT with OPTIONS   #
# CAT_IN       : /media/psf/Home/Desktop/ResearchCodes/Projects/Photoz/LePHARE_Fitting_Fortran/magnitudes.sed
# CAT_OUT      : /media/psf/Home/Desktop/ResearchCodes/Projects/Photoz/LePHARE_Fitting_Fortran/powerlaw_model_fitting/powerlaw_GAL.out
# CAT_LINES     :        -99        -99
# PARA_OUT     : /media/psf/Home/Desktop/ResearchCodes/Projects/Photoz/LePHARE_Fitting_Fortran/zphot_output.para
# INP_TYPE     : M
# CAT_FMT[0:MEME 1:MMEE]:      0
# CAT_MAG      : AB
# ZPHOTLIB     : PL_some_GR+UV STAR_GR+UV 
# ADD_EMLINES  : NO
# ERR_SCALE    :  0.100  0.100  0.100  0.100  0.100  0.100  0.100  0.100  0.100  0.100
# ERR_FACTOR   :  1.000
# BD_SCALE     :          0
# GLB_CONTEXT  :          0
# CHI2_RM_BD   :  10000.00 10000.00
# Z_RANGE      :    0.000   99.990
# EBV_RANGE  :    0.000    9.000
# DZ_WIN       :  0.500
# MIN_THRES    :  0.100
# MASS_SCALE   :    0.000   0.000
# MAG_ABS      :   -8.000 -30.000


CompletedProcess(args='$LEPHAREDIR/source/zphota -c /media/psf/Home/Desktop/ResearchCodes/Projects/Photoz/LePHARE_Fitting_Fortran/zphot_powerlaw.para', returncode=0)

In [26]:
# final cleaning spec files

src_dir = root_dir
dst_dir = out_dir

dst_dir.mkdir(parents=True, exist_ok=True)

for p in src_dir.glob("I*.spec"):
    target = dst_dir / p.name
    p.replace(target) 

# Extract fitting results

Convert the LePHARE fitting output to LaTeX format.

Notes for running the extraction script:

- The columns in the `.out` file must match the `name` list used in the script. To preserve this mapping, do not modify `zphot_output.para`.
- The model list files must be present: `galaxy_library.csv` and `power-law_library.csv`. If you change the template set used for fitting, update these files accordingly.
- This script works only with the Fortran version of LePHARE.


In [29]:
# Set up file paths

root_dir = Path(".").resolve()

# galaxy model output folder
galaxy_out_path = root_dir / "galaxy_model_fitting"

# powerlaw model output folder
powerlaw_out_path = root_dir / "powerlaw_model_fitting"

# the output files
files = [
    galaxy_out_path / "galaxy_GAL.out", 
    powerlaw_out_path / "powerlaw_GAL.out"
        ]

# the library path
# same as root_dir in default
library_path = Path(".").resolve()

In [30]:
for i in files:

    if "galaxy" in i.name:
        identifier = "galaxy"
        library = pd.read_csv(library_path / "galaxy_library.csv" , sep = ",")
    else:
        identifier = "powerlaw"
        library = pd.read_csv(library_path / "power-law_library.csv", sep = ",")

    df = pd.read_csv(i, delim_whitespace = True,
                     skiprows = 55,
                     names = ["IDENT","Z_BEST", "Z_BEST68_LOW", "Z_BEST68_HIGH", 
                              "Z_ML", "CHI_BEST", "MOD_BEST", "EXTLAW_BEST", 
                              "EBV_BEST", "PDZ_BEST", "SCALE_BEST", "DIST_MOD_BEST", 
                              "NBAND_USED", "Z_SEC", "CHI_SEC", "MOD_SEC", 
                              "Z_QSO", "CHI_QSO", "MOD_QSO", "MOD_STAR", 
                              "CHI_STAR", "CONTEXT", "ZSPEC"])

    upper_limits = df["Z_BEST68_HIGH"]- df["Z_BEST"]
    lower_limits = df["Z_BEST68_LOW"] - df["Z_BEST"]
    source_num = len(df.index)

    if os.path.exists(f"{identifier}_output.csv") != True:
        output_file = open(f"{identifier}_output.csv","w")
        output_file.write("IdentNum\tPhot_z\tCHI\tP_z\tModelNum\tModel")
        output_file.close()
    else:
        os.remove(f"{identifier}_output.csv")

    for i in np.arange(source_num):
        output_file = open(f"{identifier}_output.csv","a")
        z_best = ('%.2f' % df["Z_BEST"][i])
        z_upper = ('%.2f' % upper_limits[i])
        z_lower = ('%.2f' % lower_limits[i])
        if z_lower == "0.00":
            z_lower = "-" + z_lower
        photo_z = "%s^{+%s}_{%s}" %(z_best, z_upper, z_lower)
        z_chi = "%.1f" %(df["CHI_BEST"][i])
        p_z = ('%.1f' % df["PDZ_BEST"][i])
        fitting_id = df["IDENT"][i]
        if df["MOD_BEST"][i] != -999:
            model_num = df["MOD_BEST"][i]
            model = library["Model"][library["Model_ID"] == model_num].item()
            if identifier == "pl_GAL":
                model = ('%.2f' % model)
        else:
            model_num = df["MOD_BEST"][i]
            model = "None"
        output_file.write(f"\n{fitting_id}\t{photo_z}\t{z_chi}\t{p_z}\t{model_num}\t{model}")
        output_file.close()
        print(f"source {fitting_id} is finished")

source 44225802 is finished
source 20024546 is finished
source 21334966 is finished
source 19402431 is finished
source 34957064 is finished
source 14321234 is finished
source 16375034 is finished
source 16106649 is finished
source 30609413 is finished
source 44111234 is finished
source 60015124 is finished
source 44225802 is finished
source 20024546 is finished
source 21334966 is finished
source 19402431 is finished
source 34957064 is finished
source 14321234 is finished
source 16375034 is finished
source 16106649 is finished
source 30609413 is finished
source 44111234 is finished
source 60015124 is finished
